In [0]:
print("new")

In [0]:
content = """# DLT Internals - Technical Discussion

_Deep-dive into how DLT pipelines work internally, captured from architecture discussions (2026-03-26)_

---

## 1. What Happens During a Dry Run?

A dry run validates the pipeline without processing any data:

| Step | What happens | Data processed? |
|---|---|---|
| 1. Parse source files | Reads all Python/SQL files, executes decorators | No |
| 2. Build DAG | Resolves dp.read() / STREAM() references to build dependency graph | No |
| 3. Validate schemas | Checks column references, types, struct field access | No |
| 4. Check sources exist | Verifies external tables/volumes actually exist | No |
| 5. Validate expectations | Confirms constraint expressions are syntactically valid | No |
| 6. Report errors | Returns any issues found | No |

No data is read, written, or moved. No checkpoints created. No tables modified.

---

## 2. DLT's Intermediate Representation (IR)

DLT doesn't just compile individual DataFrames (Spark does that) and doesn't execute them (that's a pipeline update). It builds an intermediate representation - a pipeline graph model - that sits between compilation and execution:

Individual Spark -> DLT Intermediate -> Actual Execution
(per DataFrame)    (whole pipeline)     (data processing)
                        ^
                   DRY RUN VALIDATES THIS

This IR contains:
- Every dataset definition (name, type, schema)
- Every flow (streaming vs batch, source to target)
- The resolved DAG (dependency edges)
- Expected schemas at each boundary
- Constraint/expectation definitions
- Checkpoint compatibility info

---

## 3. What DLT Knows That Spark Doesn't

Spark validates one DataFrame at a time. DLT validates the entire pipeline as a system.

| What needs validation | Spark knows? | DLT knows? | Why Spark can't |
|---|---|---|---|
| Does dp.read("gharchive_bronze") exist? | No | Yes | Bronze is defined in a different function/file |
| Is the upstream a streaming table or MV? | No | Yes | Spark doesn't know what type another @dp.table produces |
| Will bronze exist before silver runs? | No | Yes | Spark has no cross-function execution ordering |
| Is checkpoint state compatible? | No | Yes | Checkpoints are outside Spark compile-time scope |
| Do all datasets form a valid DAG? | No | Yes | Spark sees individual DataFrames, not the graph |

### The Core Problem: Deferred Resolution

dp.read("gharchive_bronze") in silver - at Spark compile time, gharchive_bronze might not exist yet (defined in another cell/file). Spark would fail with TABLE_OR_VIEW_NOT_FOUND. DLT solves this by scanning all files first to discover every dataset definition.

### Code vs Catalog Validation

Critical gap: Spark validates against what's in the catalog (materialized tables). DLT validates against what's in the code (source of truth).

Example: If you rename a column in bronze code but silver still references the old name:
- Spark: Silver compile PASSES (resolves against existing materialized table with old schema)
- DLT: Silver dry run FAILS (resolves against new bronze code definition, catches the mismatch)

Analogy: Spark is a compiler checking one file. DLT dry run is a linker checking the whole program.

---

## 4. DLT Features Impose Constraints That Don't Exist in Plain Spark

| DLT feature | Constraint it imposes | What dry run validates |
|---|---|---|
| Named datasets (@dp.table(name="X")) | Names must be unique across all files | No duplicate dataset names |
| Cross-dataset reads (dp.read("X")) | Target must exist as a defined dataset | All references resolve to known datasets |
| Dataset types (ST vs MV) | Read semantics must match type | Read mode matches source type |
| DAG structure | No circular dependencies | Graph is acyclic |
| Expectations | Expressions must be valid against dataset schema | Expressions parse and columns exist |
| Schema contract | If explicit schema defined, output must match | Schema compatibility |
| Append flows | Target must be a streaming table | Target type is correct |
| Auto CDC | Keys, sequence_by columns must exist | Column references valid |

None of these constraints exist in plain Spark. The constraints are a consequence of the declarative model.

---

## 5. Could You Build This Yourself?

### DAG Resolution - Yes (~50-100 lines)

Approach A - AST parsing (static analysis, no Spark needed)
Approach B - Spark logical plan inspection (df._jdf.queryExecution().analyzed())

### What's Hard to Replicate

| Feature | DIY (effort) | DLT |
|---|---|---|
| Find dependencies | ~50 lines Python | Automatic |
| Topological sort | Standard graph algorithm | Built in |
| Detect cycles | Check for back edges | Reports error |
| Checkpoint management per node | Significant engineering | Automatic |
| Partial refresh (run subset) | Complex state management | Built-in UI |
| Retry failed nodes | Custom retry framework | Automatic (2+ retries) |
| Full refresh with state reset | Manual checkpoint deletion | One button |

The DAG resolution is straightforward. The hard part is operationalizing it.

---

## 6. Why the Pipeline Abstraction Exists

Without DLT, every team rebuilds:
1. A config system for checkpoint paths and output tables
2. A DAG scheduler for table dependencies
3. A state tracker for processed data
4. A retry framework for transient failures
5. A refresh mechanism to reset and reprocess
6. A monitoring system for run history

Same reason Kubernetes exists over Docker - operational burden of managing state, dependencies, and environments at scale exceeds what's reasonable in application code.

In plain Spark, there is no pipeline boundary - each script is independent, and nobody validates the system as a whole until runtime failures tell you something is wrong.
"""

path = "/Workspace/Users/prateek.panjla@ltimindtree.com/dbr_data_proc/DLT_DISCUSSION.md"
dbutils.fs.put(path.replace("/Workspace", "file:/Workspace"), content, overwrite=True)
print(f"Done - written to {path}")